In [1]:
import pandas as pd
import ta
import os

In [7]:
os.chdir("..")

TICKERS = ["SPY", "QQQ", "AAPL"]
RAW_PATH = os.path.join("data", "raw")
PROCESSED_PATH = os.path.join("data", "processed")

In [8]:
for ticker in TICKERS:
    df = pd.read_csv(os.path.join(RAW_PATH, f"{ticker}.csv"), index_col="Date", parse_dates=True)

    rows_before = len(df)
    ## relative strength index
    df["RSI"] = ta.momentum.RSIIndicator(df["Close"], window=14).rsi()

    ##moving average convergence divergence
    macd = ta.trend.MACD(df["Close"], window_slow=26, window_fast=12, window_sign=9)

    df["MACD"] = macd.macd()
    df["MACD_Signal"] = macd.macd_signal()
    df["MACD_Hist"] = macd.macd_diff()

    # Bollinger Bands
    bb = ta.volatility.BollingerBands(df["Close"], window=20, window_dev=2)

    df["BB_Upper"] = bb.bollinger_hband()
    df["BB_Middle"] = bb.bollinger_mavg()
    df["BB_Lower"] = bb.bollinger_lband()

    # Simple Moving Averages
    df["SMA_20"] = ta.trend.SMAIndicator(df["Close"], window=20).sma_indicator()
    df["SMA_50"] = ta.trend.SMAIndicator(df["Close"], window=50).sma_indicator()
    df["SMA_200"] = ta.trend.SMAIndicator(df["Close"], window=200).sma_indicator()

    # Exponential Moving Averages
    df["EMA_12"] = ta.trend.EMAIndicator(df["Close"], window=12).ema_indicator()
    df["EMA_26"] = ta.trend.EMAIndicator(df["Close"], window=26).ema_indicator()

    # ATR - Average True Range (volatility)
    df["ATR"] = ta.volatility.AverageTrueRange(
        df["High"], df["Low"], df["Close"], window=14
    ).average_true_range()

    # OBV - On Balance Volume
    df["OBV"] = ta.volume.OnBalanceVolumeIndicator(
        df["Close"], df["Volume"]
    ).on_balance_volume()
    ##Schochastic Oscillator
    stoch = ta.momentum.StochasticOscillator(
        df["High"], df["Low"], df["Close"], window=14, smooth_window=3
    )
    df["Stoch_K"] = stoch.stoch()
    df["Stoch_D"] = stoch.stoch_signal()

    # Williams %R
    df["Williams_R"] = ta.momentum.WilliamsRIndicator(
        df["High"], df["Low"], df["Close"], lbp=14
    ).williams_r()

    # Rate of Change
    df["ROC"] = ta.momentum.ROCIndicator(df["Close"], window=12).roc()

    df = df.dropna()

    rows_after = len(df)

    print(f"{ticker}: dropped {rows_before - rows_after} rows, {rows_after} rows remaining")

    os.makedirs(PROCESSED_PATH, exist_ok=True)
    df.to_csv(os.path.join(PROCESSED_PATH, f"{ticker}_features.csv"))
    print(f"Saved to {PROCESSED_PATH}/{ticker}_features.csv")


SPY: dropped 199 rows, 2317 rows remaining
Saved to data\processed/SPY_features.csv
QQQ: dropped 199 rows, 2317 rows remaining
Saved to data\processed/QQQ_features.csv
AAPL: dropped 199 rows, 2317 rows remaining
Saved to data\processed/AAPL_features.csv
